In [1]:
# LLMS
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

llm = ChatGroq(model="openai/gpt-oss-120b")
light_llm = init_chat_model(
    model="meta-llama/llama-3.3-70b-instruct", 
    model_provider="openrouter",
)
reasoning_llm = init_chat_model(
    model="deepseek/deepseek-r1",
    model_provider="openrouter"
)


In [2]:
from pageindex import PageIndexClient
from dotenv import load_dotenv
load_dotenv()

# 1. create pageindex client
pageindexClient = PageIndexClient(
    index="cloud", 
    chat="deepseek/deepseek-r1:free"
)


In [3]:
print(pageindexClient.list_documents())

{'documents': [{'id': 'pi-cmuvhx9co005p0bmb1jy3i7uk', 'name': 'AI Engineering Guidebook.pdf', 'description': 'This document is a comprehensive guide to AI Engineering, detailing system design patterns, LLM architectures, prompt and context engineering, fine-tuning techniques, RAG systems, AI agents, model optimization, evaluation frameworks, and production deployment strategies.', 'status': 'completed', 'createdAt': '2026-10-05T17:00:36.907000', 'pageNum': 384, 'folderId': None, 'metadata': {'thumbnailKey': 'cmuu3qvnl000004i94r3h464c/pageindex/AI Engineering Guidebook_thumbnail.jpg', 'thumbnailGeneratedAt': '2026-10-05T17:00:38.817167Z'}, 'features': {'blockReference': True, 'blockReferenceStatus': 'completed'}, 'path': None}], 'total': 1, 'limit': 50, 'offset': 0}


In [4]:
# 2. upload and index page(upload pdf ->pageindex uses llm to read doc structure -> builds hierarchical tree index -> returns docId)

pdf_path = "AI Engineering Guidebook.pdf"
print(f"uploading{pdf_path}")
# res = pageindexClient.submit_document(pdf_path)
doc_id = pageindexClient.get_document_id(pdf_path)

print("Uploaded")
print(f"Doc Id {doc_id}")

uploadingAI Engineering Guidebook.pdf
Uploaded
Doc Id pi-cmuvhx9co005p0bmb1jy3i7uk


In [5]:
import json
# 3. Fetch the full tree
tree_res = pageindexClient.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_res.get("result", [])

print("tree_res: ", tree_res)
print("pageindex_tree: ", pageindex_tree)

print(f"top Level Sections : {len(pageindex_tree)}")
print("First Node")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=3))

tree_res:  {'doc_id': 'pi-cmuvhx9co005p0bmb1jy3i7uk', 'status': 'completed', 'retrieval_ready': True, 'result': [{'title': 'AI Engineering', 'node_id': '0000', 'start_index': 1, 'end_index': 384, 'summary': 'This section covers comprehensive concepts and system design patterns in AI Engineering, beginning with foundational LLM architectures, text generation strategies, and reading guidance. It explores prompt engineering techniques (like Chain of Thought and JSON prompting) and fine-tuning methods, including parameter-efficient techniques such as LoRA and reinforcement learning approaches like GRPO. The text details Retrieval-Augmented Generation (RAG) and its advanced variations, alongside the transition to context engineering and the architecture, memory types, design patterns, and orchestration protocols of AI agents. Furthermore, it addresses model optimization (quantization, pruning, distillation, and KV caching), evaluation frameworks (LLM-as-a-judge, safety red teaming), high-pe

In [6]:
# Pretty print the whole tree

def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    
    for node in nodes:
        prefix = " "*indent + ("L " if indent > 0 else "")
        page = node.get("start_index", "?")

        print(f"{prefix}[{node['node_id']}] {node['title']} (p. {page})")

        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)


print("Full Document Structure:\n")
print_tree(pageindex_tree, 6)

Full Document Structure:

      L [0000] AI Engineering (p. 1)
       L [0001] AI Engineering (intro) (p. 1)
       L [0002] How to make the most out of this book and your time? (p. 2)
        L [0003] How to make the most out of this book and your time? (intro) (p. 2)
        L [0004] AI Engineering (p. 3)
         L [0005] AI Engineering (intro) (p. 3)
         L [0006] What is an LLM? (p. 8)
         L [0007] Need for LLMs (p. 11)
         L [0008] What makes an LLM 'large'? (p. 13)
         L [0009] How are LLMs built? (p. 15)
         L [0010] How to train LLM from scratch? (p. 19)
         L [0011] How do LLMs work? (p. 24)
         L [0012] 7 LLM Generation Parameters (p. 30)
         L [0013] 1) Max tokens (p. 30)
         L [0014] 2) Temperature (p. 31)
         L [0015] 3) Top-k (p. 31)
         L [0016] 4) Top-p (nucleus sampling) (p. 31)
         L [0017] 5) Frequency penalty (p. 32)
         L [0018] 6) Presence penalty (p. 32)
         L [0019] 7) Stop sequences (p. 33)
 

In [7]:
# Count the total number of nodes in a tree

def count_nodes(nodes):
    total = len(nodes)
    for node in nodes:
        if node.get("nodes"):
            total += count_nodes(node["nodes"])
    return total
print("Total Nodes in a tree ")
count_nodes(pageindex_tree)

Total Nodes in a tree 


158

In [8]:
import os
from openai import OpenAI

# 1. Initialize the client with Groq's endpoint and your Groq API key
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY") 
)



In [48]:
import json


# LLM Tree Search 
def llm_search_tree(query, tree, model) -> dict:
    """
    Sends query + doc tree to llm 
    llm reasons over structure and returns reterival nodes
    """
    def compress(nodes):
        out = []
        for node in nodes:
            entry = {
                "node_id": node["node_id"],
                "title": node["title"],
                # "page_index": node.get("page_index", "?"),
                # "summary": node.get("text", "")[:50]
            }
            if node.get("nodes"):
                entry["children"] = compress(node["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
    Your task: identify which node IDs most likely contain the answer to the query.
    Think step-by-step about which sections are relevant.

    Query: {query}

    Document Tree:
    {json.dumps(compressed_tree, separators=(",", ":"))}

    Reply ONLY in this exact JSON format:

    "node_list": ["node_id1", "node_id2"]
    """
    # 2. Submit the chat completion request
    response = client.chat.completions.create(
        model=model, 
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)
    




In [49]:
# Perform a sample test

query = "Will it be easy to learn finetuning after RAG"
model="qwen/qwen3.8-27b"


print(f"Query : {query}")

res = llm_search_tree(query, pageindex_tree, model)
print(f"LLM Reasoning ...")
print(res.get("thinking", "N/A"))
print("Selected Node Ids\n\n")
print(res.get("node_list", []))

Query : Will it be easy to learn finetuning after RAG
LLM Reasoning ...
N/A
Selected Node Ids


['0061', '0065']


In [50]:
# Find Nodes by id
def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    # Convert to a set for O(1) lightning-fast lookups
    target_set = set(target_ids) 
    
    for node in tree:
        if node["node_id"] in target_set:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

reterived_nodes = find_nodes_by_ids(pageindex_tree, res.get("node_list", []))

In [51]:
def generate_answer(query: str, nodes: list, model: str = "openai/gpt-oss-120b"):
    if not nodes:
        print("No relevant Sections found")
        return
    # build context string from reterived nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[section: {node['title']} | Page. {node.get('page_index', node.get('start_index', '?'))}]\n"
            f"{node.get('text')}"
        )
    context = "\n\n".join(context_parts)
    prompt = f"""You are an expert document analyst.
    Answer the question using ONLY the provided context.
    For every claim you make, cite the section title and page number in parentheses.
    Be concise and precise.

    Question: {query}

    Context:
    {context}
    Provide your final response as a valid JSON object.
    Answer: """
    # 2. Submit the chat completion request
    response = client.chat.completions.create(
        model=model, 
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)


In [38]:
import json

# 1. Dynamically pull the exact active models available on your account right now
try:
    available_models = [m.id for m in client.models.list().data]
    # Filter out whisper audio models or specific embeddings models if any exist
    text_models = [m for m in available_models if "whisper" not in m.lower() and "embed" not in m.lower()]
    
    # Grab the first available production text model slug automatically
    active_model = text_models[0] if text_models else "llama-3.3-70b-versatile"
    print(f"✅ Found active Groq text models: {text_models}")
    print(f"🚀 Using dynamic model target: '{active_model}'")
except Exception as e:
    # Fallback if the client list endpoint hits a rate block
    active_model = "llama-3.3-70b-versatile" 
    print(f"⚠️ Could not list models dynamically ({e}). Falling back to baseline slug.")



✅ Found active Groq text models: ['openai/gpt-oss-120b', 'meta-llama/llama-prompt-guard-2-22m', 'meta-llama/llama-prompt-guard-2-86m', 'canopylabs/orpheus-arabic-saudi', 'qwen/qwen3.8-27b', 'openai/gpt-oss-20b', 'canopylabs/orpheus-v1-english', 'openai/gpt-oss-safeguard-20b', 'allam-2-7b']
🚀 Using dynamic model target: 'openai/gpt-oss-120b'


In [52]:
generate_answer(query, reterived_nodes, "openai/gpt-oss-120b")

{'answer': 'The text does not say that fine‑tuning becomes easy after using RAG; it simply contrasts the two methods, noting that RAG avoids the need for fine‑tuning (Prompting vs. RAG vs. Finetuning?, p.125) and that fine‑tuning is a distinct technique with its own considerations (Full-model Fine-tuning vs. LoRA vs. RAG, p.135).'}